# 01. Explorar variáveis

Profile, blocking Splink, cobertura GT. Fonética básica em `*_phon`; agressiva em `*_phon_sv` se `USE_PHONETIC_STRIP_VOWELS=True`.


In [ ]:
import sys, json
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

import pandas as pd
from config import SPLINK_SETTINGS_DRAFT, USE_PHONETIC_STRIP_VOWELS, get_connection, print_paths, require_tables

print_paths()
con = get_connection()
require_tables(con, ['registro_unificado', 'ground_truth_clusters'], notebook_origem='00')


In [ ]:
VARS = [
    'nome_completo', 'primeiro_nome', 'nome_meio', 'ultimo_nome', 'nome_mae',
    'sexo', 'data_nascimento', 'cep', 'uf', 'cpf_norm',
    'nome_completo_phon', 'primeiro_nome_phon', 'ultimo_nome_phon',
]
if USE_PHONETIC_STRIP_VOWELS:
    VARS += ['nome_completo_phon_sv', 'primeiro_nome_phon_sv', 'ultimo_nome_phon_sv']

rows = []
for col in VARS:
    if col not in con.execute('SELECT * FROM registro_unificado LIMIT 0').df().columns:
        continue
    rows.append(con.execute(f'''
        SELECT '{col}' AS coluna, origem, COUNT(*) AS n,
            SUM(CASE WHEN {col} IS NOT NULL AND TRIM(CAST({col} AS VARCHAR)) <> '' THEN 1 ELSE 0 END) AS preenchido
        FROM registro_unificado GROUP BY origem
    ''').df())
profile = pd.concat(rows, ignore_index=True)
profile['pct'] = (profile['preenchido'] / profile['n'] * 100).round(1)
profile


In [ ]:
# KPI nome_mae por origem (CPF direto vs Censo inferido)
con.execute('''
SELECT origem,
    COUNT(*) AS n,
    SUM(CASE WHEN nome_mae IS NOT NULL AND TRIM(nome_mae) <> '' THEN 1 ELSE 0 END) AS com_mae,
    ROUND(100.0 * SUM(CASE WHEN nome_mae IS NOT NULL AND TRIM(nome_mae) <> '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_mae
FROM registro_unificado GROUP BY 1
''').df()


In [ ]:
gt_cov = con.execute('''
WITH pairs AS (
    SELECT g1.unique_id AS id_censo, g2.unique_id AS id_cpf
    FROM ground_truth_clusters g1
    JOIN ground_truth_clusters g2 ON g1.cluster = g2.cluster AND g1.cluster LIKE 'gt_%'
    WHERE g1.unique_id LIKE 'censo_%' AND g2.unique_id LIKE 'cpf_%'
),
in_stack AS (
    SELECT p.*,
        EXISTS (SELECT 1 FROM registro_unificado r WHERE r.unique_id = p.id_censo) AS censo_ok,
        EXISTS (SELECT 1 FROM registro_unificado r WHERE r.unique_id = p.id_cpf) AS cpf_ok
    FROM pairs p
)
SELECT COUNT(*) AS n_pares_coorte,
    SUM(CASE WHEN censo_ok AND cpf_ok THEN 1 ELSE 0 END) AS n_pares_no_subset,
    ROUND(100.0 * SUM(CASE WHEN censo_ok AND cpf_ok THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 2) AS pct_cobertura
FROM in_stack
''').df()
gt_cov


In [ ]:
from splink import DuckDBAPI, block_on
from splink.exploratory import profile_columns, n_largest_blocks

df = con.execute('''
    SELECT r.*, g.cluster FROM registro_unificado r
    LEFT JOIN ground_truth_clusters g ON r.unique_id = g.unique_id
''').df()

SAMPLE_N = min(50_000, len(df))
df_sample = df.sample(n=SAMPLE_N, random_state=42) if len(df) > SAMPLE_N else df
db_api = DuckDBAPI()
profile_columns(df_sample, db_api, column_expressions=['primeiro_nome', 'ultimo_nome', 'nome_completo_phon', 'substr(cep,1,5)', 'data_nascimento'])

for rule in [
    block_on('substr(primeiro_nome,1,3)', 'substr(ultimo_nome,1,4)'),
    block_on('ultimo_nome', 'data_nascimento'),
    block_on('substr(cep,1,5)', 'primeiro_nome'),
    block_on('nome_mae', 'data_nascimento'),
]:
    n_largest_blocks(table_or_tables=df_sample, blocking_rule=rule, db_api=db_api, n=5)


In [ ]:
draft = {
    'link_type': 'dedupe_only',
    'cpf_in_comparisons': False,
    'phonetic_basic_columns': ['nome_completo_phon', 'primeiro_nome_phon', 'ultimo_nome_phon'],
    'phonetic_strip_vowels': USE_PHONETIC_STRIP_VOWELS,
    'blocking_rules': [
        'substr(primeiro_nome,1,3) + substr(ultimo_nome,1,4)',
        'ultimo_nome + data_nascimento',
        'nome_mae + data_nascimento',
    ],
}
SPLINK_SETTINGS_DRAFT.parent.mkdir(parents=True, exist_ok=True)
SPLINK_SETTINGS_DRAFT.write_text(json.dumps(draft, indent=2, ensure_ascii=False))
print('Salvo:', SPLINK_SETTINGS_DRAFT)
con.close()
